## Performance Bounds: The Roofline Model, Arithmetic Intensity, and Compute vs. Memory Bound Kernels

Every optimization decision in GPU programming boils down to one fundamental question:

"Is this operation waiting on math, or is it waiting on memory?"

If an operation is waiting on memory, upgrading your Tensor Cores or optimizing mathematical formulas will yield 0% speedup. Conversely, if it is waiting on math, optimizing memory layouts will yield 0% speedup.

The Roofline Model is the quantitative framework used to identify where any kernel hits its performance wall.

### Roofline Model

## 1. Arithmetic Intensity

Arithmetic Intensity, also called operational intensity, measures how much useful computation is performed per byte of memory traffic.

$$
I = \frac{\text{Total Floating-Point Operations (FLOPs)}}{\text{Total Memory Transferred (Bytes from HBM/DRAM)}}
$$

- The numerator is the number of arithmetic operations such as additions, multiplications, or fused multiply-accumulate operations.
- The denominator is the number of bytes moved from global memory.

### Intuition

- High arithmetic intensity means the kernel does a lot of work for each byte it reads or writes.
- Low arithmetic intensity means the kernel spends most of its time moving data rather than computing.

### Simple comparison

- A kernel that loads 1 byte and performs 10,000 FLOPs is compute-heavy.
- A kernel that loads 1,000 bytes and performs 1 FLOP per byte is memory-heavy.

---

## 2. The Roofline Model

A GPU has two independent hardware ceilings:

- Peak compute throughput: the maximum arithmetic rate the SMs and Tensor Cores can sustain.
- Peak memory bandwidth: the maximum rate at which data can move over the HBM bus.

The attainable performance of a kernel is bounded by:

$$
P = \min\left(\text{Peak Compute Throughput},\; I \times \text{Peak Memory Bandwidth}\right)
$$

```text
Performance (TFLOP/s)
  ▲
  │                       PEAK COMPUTE CEILING
  │                    ┌───────────────────────────────────────────────
  │                   /│
  │                  / │
  │                 /  │   COMPUTE-BOUND REGION
  │                /   │
  │  MEMORY-    /      │
  │   BOUND    /       │
  │  REGION   /        │
  │          /         │
  │         /          │
  │        /           │
  │       /            │
  └─────┴──────────────┴─────────────────────────────────────────────►
       0            I_crit (Machine Balance)           Arithmetic Intensity (FLOPs / Byte)
```

The knee of the curve is the critical balance point:

$$
I_{\text{crit}} = \frac{\text{Peak Compute Throughput (FLOP/s)}}{\text{Peak Memory Bandwidth (Bytes/s)}}
$$

---

## 3. Concrete Example: NVIDIA H100 SXM5

For BF16 Tensor Core operations on an H100:

- Peak BF16 compute throughput is about 989 TFLOPS.
- Peak HBM bandwidth is about 3.35 TB/s.

So the machine balance is approximately:

$$
I_{\text{crit}} \approx \frac{989 \times 10^{12}}{3.35 \times 10^{12}} \approx 295.2 \text{ FLOPs / Byte}
$$

### Engineering takeaway

- If a kernel performs fewer than about 295 FLOPs per byte, it is memory-bound.
- If it performs more than about 295 FLOPs per byte, it is compute-bound.

---

## 4. Case Studies: Common LLM Operations

### Case 1: Vector addition, ReLU, or RMSNorm

```text
CASE 1: VECTOR ADDITION / RELU / RMSNORM

Operation: y = x + b   (Vector of size N)
Memory:    Load x (2N bytes), Load b (2N bytes), Write y (2N bytes) = 6N bytes
Math:      N additions = N FLOPs

Arithmetic Intensity: I = N / 6N = 0.167 FLOPs / Byte
Verdict: HEAVILY MEMORY-BOUND
```

This kind of kernel is usually limited by memory traffic. Fusing adjacent operations can help keep data in registers or shared memory instead of sending it back to HBM.

### Case 2: Matrix-vector multiplication (GEMV)

```text
CASE 2: MATRIX-VECTOR MULTIPLICATION (GEMV)

Operation: y = W * x
Memory:    Load W (2 * M * K bytes), Load x (2K bytes), Write y (2M bytes)
Math:      M * K multiply-accumulate operations = 2 * M * K FLOPs

Arithmetic Intensity: I ≈ 1.0 FLOP / Byte
Verdict: EXTREMELY MEMORY-BOUND
```

This is especially common during single-token decoding, where the GPU must stream weights from HBM for each token.

### Case 3: Matrix-matrix multiplication (GEMM)

```text
CASE 3: MATRIX-MATRIX MULTIPLICATION (GEMM)

Operation: Y = X * W
Memory:    Load X + Load W + Write Y
Math:      2 * B * M * K FLOPs

Example with B = 512, M = K = 4096:
Memory:    ≈ 41.9 MB
Math:      ≈ 17.18 GFLOPs

Arithmetic Intensity: I ≈ 410 FLOPs / Byte
Verdict: COMPUTE-BOUND
```

This is why prompt processing can become compute-bound: batching many tokens lets the GPU reuse matrix data across many operations and keep the Tensor Cores busy.
